<!-- 🔒 FULL GAME OVERLAY -->
<div id="gameOverlay" style="
  position:fixed;top:0;left:0;width:100%;height:100%;
  background:linear-gradient(135deg,#0b1020,#111827);
  color:#fff;font-family:'Segoe UI',Arial,sans-serif;
  z-index:999999;display:flex;flex-direction:column;
  align-items:center;justify-content:center;overflow:hidden;
">

  <!-- STAGE 1: MEMORY MATCH -->
  <div id="memoryStage" style="text-align:center;">
    <h1 style="color:#00d4ff;font-size:40px;margin:0 0 10px;text-shadow:0 0 20px #00d4ff;">
      🎮 FRUIT MATCH — STAGE 1
    </h1>
    <p style="color:#aaa;font-size:18px;margin:0 0 20px;">
      Match all the fruit pairs to reveal the maze!
    </p>

    <div style="
      background:rgba(255,255,255,0.06);
      padding:20px;border-radius:15px;
      border:2px solid #00d4ff;
      box-shadow:0 0 30px rgba(0,212,255,0.3);
      display:inline-block;
    ">
      <div style="margin-bottom:15px;font-size:18px;">
        <span>⭐ Matches: <span id="mmMatches" style="color:#ffd700;">0</span>/4</span>
        <span style="margin-left:30px;">🔄 Moves: <span id="mmMoves" style="color:#00d4ff;">0</span></span>
      </div>

      <div id="mmGrid" style="
        display:grid;
        grid-template-columns:repeat(4,80px);
        gap:12px;
        justify-content:center;
      "></div>

      <div id="mmWin" style="display:none;margin-top:20px;">
        <h2 style="color:#00ff88;font-size:26px;margin:0;">🎉 Nice! Maze Unlocked! 🎉</h2>
        <p style="color:#fff;margin:8px 0 0;">Get all fruits to the center in Stage 2.</p>
      </div>
    </div>
  </div>

  <!-- STAGE 2: MAZE -->
  <div id="mazeStage" style="display:none;text-align:center;">
    <h1 style="color:#00ff88;font-size:40px;margin:0 0 10px;text-shadow:0 0 20px #00ff88;">
      🧩 FRUIT MAZE — STAGE 2
    </h1>
    <p id="mazeDirections" style="color:#aaa;font-size:18px;margin:0 0 15px;">
      Click a fruit to select it (it glows), then use the arrow keys to move it. Get ALL fruits to the center tile.
    </p>

    <div style="
      display:inline-block;
      padding:15px;
      background:rgba(255,255,255,0.06);
      border-radius:15px;
      border:2px solid #00ff88;
      box-shadow:0 0 30px rgba(0,255,136,0.3);
    ">
      <div style="margin-bottom:10px;font-size:16px;">
        <span>🍉 🍓 🍍 🍒 → all must reach the center.</span>
      </div>
      <div id="mazeGrid" style="
        display:grid;
        grid-template-columns:repeat(15,24px);
        grid-template-rows:repeat(15,24px);
        gap:2px;
        justify-content:center;
      "></div>
      <div id="mazeWin" style="display:none;margin-top:15px;">
        <h2 style="color:#00ff88;font-size:26px;margin:0;">🎉 You guided all fruits to the center! 🎉</h2>
        <p style="color:#fff;margin:8px 0 0;">Lesson unlocked. You can replay the game anytime.</p>
      </div>
    </div>
  </div>
</div>

<!-- 🔁 PLAY AGAIN BUTTON (top-right, after full win) -->
<button id="playAgainBtn" onclick="restartFullGame()" style="
  position:fixed;top:20px;right:20px;z-index:999998;
  padding:10px 20px;font-size:16px;border:none;
  background:#00d4ff;color:#000;border-radius:10px;
  cursor:pointer;font-weight:bold;display:none;
  box-shadow:0 4px 10px rgba(0,0,0,0.3);
">
  🍉 Play Matching & Maze Again
</button>

<script>
// ====== GLOBAL CONTROL ======
function hidePageExceptGame() {
  document.body.querySelectorAll('*:not(#gameOverlay):not(#playAgainBtn)').forEach(el => {
    el.style.visibility = 'hidden';
  });
  document.body.style.overflow = 'hidden';
}
function showPage() {
  document.body.querySelectorAll('*').forEach(el => {
    el.style.visibility = 'visible';
  });
  document.body.style.overflow = 'auto';
}

// ====== STAGE 1: MEMORY MATCH ======
const mmFruits = ['🍉','🍓','🍍','🍒'];
let mmCards = [];
let mmFlipped = [];
let mmMatches = 0;
let mmMoves = 0;

function initMemoryMatch() {
  const grid = document.getElementById('mmGrid');
  const winBox = document.getElementById('mmWin');
  winBox.style.display = 'none';
  mmMatches = 0;
  mmMoves = 0;
  document.getElementById('mmMatches').textContent = mmMatches;
  document.getElementById('mmMoves').textContent = mmMoves;

  const pairData = [];
  mmFruits.forEach((f, idx) => {
    pairData.push({ id: idx, emoji: f });
    pairData.push({ id: idx, emoji: f });
  });

  // shuffle
  for (let i = pairData.length - 1; i > 0; i--) {
    const j = Math.floor(Math.random() * (i + 1));
    [pairData[i], pairData[j]] = [pairData[j], pairData[i]];
  }

  mmCards = pairData;
  mmFlipped = [];
  grid.innerHTML = '';

  pairData.forEach((card, index) => {
    const cardEl = document.createElement('div');
    cardEl.style.cssText = `
      width:80px;height:80px;
      background:linear-gradient(145deg,#1f2937,#111827);
      border-radius:12px;display:flex;align-items:center;
      justify-content:center;cursor:pointer;font-size:40px;
      box-shadow:0 4px 10px rgba(0,0,0,0.4);
      transition:transform 0.15s, background 0.15s;
      color:#fff;
    `;
    cardEl.dataset.index = index;
    cardEl.textContent = '❓';

    cardEl.onmouseover = () => cardEl.style.transform = 'scale(1.05)';
    cardEl.onmouseout = () => cardEl.style.transform = 'scale(1)';

    cardEl.onclick = () => flipMemoryCard(cardEl);
    grid.appendChild(cardEl);
  });
}

function flipMemoryCard(cardEl) {
  const idx = parseInt(cardEl.dataset.index);
  if (mmFlipped.length >= 2) return;
  if (mmFlipped.some(c => c.idx === idx)) return;

  const card = mmCards[idx];
  cardEl.textContent = card.emoji;
  cardEl.style.background = 'linear-gradient(145deg,#00d4ff,#0099cc)';
  mmFlipped.push({ idx, cardEl, id: card.id });

  if (mmFlipped.length === 2) {
    mmMoves++;
    document.getElementById('mmMoves').textContent = mmMoves;
    const [c1, c2] = mmFlipped;
    if (c1.id === c2.id) {
      mmMatches++;
      document.getElementById('mmMatches').textContent = mmMatches;
      c1.cardEl.style.background = 'linear-gradient(145deg,#00ff88,#00cc6a)';
      c2.cardEl.style.background = 'linear-gradient(145deg,#00ff88,#00cc6a)';
      mmFlipped = [];
      if (mmMatches === mmFruits.length) {
        document.getElementById('mmWin').style.display = 'block';
        setTimeout(() => {
          // move to maze stage
          document.getElementById('memoryStage').style.display = 'none';
          document.getElementById('mazeStage').style.display = 'block';
          initMaze();
        }, 800);
      }
    } else {
      setTimeout(() => {
        c1.cardEl.textContent = '❓';
        c2.cardEl.textContent = '❓';
        c1.cardEl.style.background = 'linear-gradient(145deg,#1f2937,#111827)';
        c2.cardEl.style.background = 'linear-gradient(145deg,#1f2937,#111827)';
        mmFlipped = [];
      }, 700);
    }
  }
}

// ====== STAGE 2: MAZE ======
const mazeLayout = [
  "###############",
  "#.............#",
  "#.#####.#####.#",
  "#.#...#.#...#.#",
  "#.#.#.#.#.#.#.#",
  "#...#...#...#.#",
  "###.#####.###.#",
  "#...#.....#...#",
  "#.###.###.###.#",
  "#.#...#.#...#.#",
  "#.#.#.#.#.#.#.#",
  "#.#...#...#...#",
  "#.#####.#####.#",
  "#.............#",
  "###############"
];

const mazeSize = 15;
const centerPos = { x: 7, y: 7 };

let fruits = [];
let selectedFruitId = null;

function initMaze() {
  const grid = document.getElementById('mazeGrid');
  const winBox = document.getElementById('mazeWin');
  winBox.style.display = 'none';
  document.getElementById('mazeDirections').style.display = 'block';

  fruits = [
    { id: 'watermelon', emoji: '🍉', x: 1,  y: 1  },
    { id: 'strawberry', emoji: '🍓', x: 13, y: 1  },
    { id: 'pineapple',  emoji: '🍍', x: 1,  y: 13 },
    { id: 'cherry',     emoji: '🍒', x: 13, y: 13 }
  ];
  selectedFruitId = null;

  grid.innerHTML = '';
  for (let y = 0; y < mazeSize; y++) {
    for (let x = 0; x < mazeSize; x++) {
      const tile = document.createElement('div');
      tile.dataset.x = x;
      tile.dataset.y = y;
      const ch = mazeLayout[y][x];

      if (ch === '#') {
        tile.style.cssText = `
          width:24px;height:24px;
          background:#111827;
          border-radius:2px;
          box-shadow:inset 0 0 4px rgba(0,0,0,0.8);
        `;
      } else {
        tile.style.cssText = `
          width:24px;height:24px;
          background:#1f2937;
          border-radius:2px;
        `;
        if (x === centerPos.x && y === centerPos.y) {
          tile.style.background = '#065f46';
          tile.style.boxShadow = '0 0 10px rgba(16,185,129,0.9)';
        }
      }

      tile.style.display = 'flex';
      tile.style.alignItems = 'center';
      tile.style.justifyContent = 'center';
      tile.style.fontSize = '18px';
      tile.style.cursor = 'pointer';

      tile.onclick = () => handleTileClick(x, y);
      grid.appendChild(tile);
    }
  }

  renderFruits();
}

function renderFruits() {
  const tiles = document.querySelectorAll('#mazeGrid div');
  tiles.forEach(t => {
    t.textContent = '';
    t.style.outline = 'none';
  });

  fruits.forEach(f => {
    const sel = Array.from(tiles).find(t => parseInt(t.dataset.x) === f.x && parseInt(t.dataset.y) === f.y);
    if (sel) {
      sel.textContent = f.emoji;
      if (selectedFruitId === f.id) {
        sel.style.outline = '2px solid #fbbf24';
        sel.style.boxShadow = '0 0 10px rgba(251,191,36,0.9)';
      } else {
        sel.style.boxShadow = (f.x === centerPos.x && f.y === centerPos.y)
          ? '0 0 10px rgba(16,185,129,0.9)'
          : 'none';
      }
    }
  });
}

function handleTileClick(x, y) {
  const fruit = fruits.find(f => f.x === x && f.y === y);
  if (!fruit) return;
  selectedFruitId = fruit.id;
  renderFruits();
}

function isWall(x, y) {
  if (x < 0 || x >= mazeSize || y < 0 || y >= mazeSize) return true;
  return mazeLayout[y][x] === '#';
}

function isOccupiedByOtherFruit(x, y, id) {
  return fruits.some(f => f.id !== id && f.x === x && f.y === y);
}

function moveSelectedFruit(dx, dy) {
  if (!selectedFruitId) return;
  const fruit = fruits.find(f => f.id === selectedFruitId);
  if (!fruit) return;

  const nx = fruit.x + dx;
  const ny = fruit.y + dy;

  if (isWall(nx, ny)) return;
  if (isOccupiedByOtherFruit(nx, ny, fruit.id)) return;

  fruit.x = nx;
  fruit.y = ny;
  renderFruits();
  checkMazeWin();
}

function checkMazeWin() {
  const allAtCenter = fruits.every(f => f.x === centerPos.x && f.y === centerPos.y);
  if (allAtCenter) {
    document.getElementById('mazeWin').style.display = 'block';
    document.getElementById('mazeDirections').style.display = 'none';
    // unlock lesson: hide overlay, show page, show replay button
    setTimeout(() => {
      document.getElementById('gameOverlay').style.display = 'none';
      showPage();
      document.getElementById('playAgainBtn').style.display = 'block';
    }, 900);
  }
}

// Arrow key controls
window.addEventListener('keydown', (e) => {
  const mazeVisible = document.getElementById('mazeStage').style.display !== 'none';
  if (!mazeVisible) return;
  if (['ArrowUp','ArrowDown','ArrowLeft','ArrowRight'].includes(e.key)) {
    e.preventDefault();
  }
  if (e.key === 'ArrowUp') moveSelectedFruit(0,-1);
  if (e.key === 'ArrowDown') moveSelectedFruit(0,1);
  if (e.key === 'ArrowLeft') moveSelectedFruit(-1,0);
  if (e.key === 'ArrowRight') moveSelectedFruit(1,0);
});

// ====== FULL GAME FLOW ======
function startFullGame() {
  hidePageExceptGame();
  document.getElementById('gameOverlay').style.display = 'flex';
  document.getElementById('memoryStage').style.display = 'block';
  document.getElementById('mazeStage').style.display = 'none';
  initMemoryMatch();
}

function restartFullGame() {
  startFullGame();
}

// Run on page load
window.addEventListener('DOMContentLoaded', startFullGame);
</script>


## What are Strings?
---
- A **string** is a sequence of characters **(letters, numbers, symbols, spaces)** enclosed in quotes.
## Examples

- ``"Hello"``

- ``"12345"``

- ``"Good Morning"``

## Challenge 1 - Cookie Clicking
Click the Cookie for a quiz!

<div id="gameArea" 
     style="position:relative;width:330px;height:230px;border:2px solid #8b5a2b;
            overflow:hidden;background:#fff3e0;display:flex;justify-content:center;
            align-items:center;">

  <div id="cookie" style="font-size:150px;cursor:pointer;">
    🍪
  </div>

</div>

<p id="question" style="font-size:18px;font-weight:bold;margin-top:10px;"></p>

<script>
// 5 IMPROVED QUESTIONS
const questions = [
  // BASIC
  "BASIC: Which one is a string: \"hello\" or hello",

  // PYTHON
  "PYTHON: In Python, which one is a valid string: 'Python is fun!' or Python is fun!",
  "PYTHON: Which one creates a multi-line string: \"\"\"text\"\"\" or \"text\"",

  // JAVASCRIPT
  "JS: In JavaScript, which one is a string: \"hello\" or hello",
  "JS: Which one makes a multi-line string: `text` or \"text\""
];

document.getElementById("cookie").onclick = () => {
  const cookie = document.getElementById("cookie");

  // 2× POP / VIBRATION EFFECT
  cookie.style.transition = "transform 0.22s ease";
  cookie.style.transform = "scale(1.5) rotate(14deg)";
  setTimeout(() => {
    cookie.style.transform = "scale(1)";
  }, 220);

  // Pick a random question
  const q = questions[Math.floor(Math.random() * questions.length)];
  document.getElementById("question").textContent = "Challenge: " + q;
};
</script>

### **Python**

In [1]:
myName = "Neil"  # We create a string variable called myName and give it the value "Neil"
myNameType = type(myName).__name__  # type(myName).__name__ returns the name of the data type: "str"

print(f"{myName} is a {myNameType} data type.")  # Prints: Neil is a str data type.

Neil is a str data type.


### **JavaScript**

In [ ]:
// JavaScript code to manage the game visibility

// Function to initialize the game
function initGame() {
    const gameContainer = document.getElementById('game-container');
    const lessonContent = document.getElementById('lesson-content');

    // Hide lesson content initially
    lessonContent.style.display = 'none';

    // Show only the game
    gameContainer.style.display = 'block';

    // Add event listener for game completion
    document.getElementById('complete-game').addEventListener('click', () => {
        // Show lesson content after game completion
        lessonContent.style.display = 'block';
        gameContainer.style.display = 'none';
    });
}

// Call the initGame function on page load
window.onload = initGame;

<IPython.core.display.Javascript object>

### Defining Strings
- In most programming languages, you can make a string with either single (') or double (") quotes:

### **Python**

In [3]:
word = "hello" # <-- Here, we use double quotes instead of single quotes
sentence = 'Python is fun!' # <-- Here, we use single quotes instead of double quotes
# Both are valid ways to define strings in Python
multiline = """This is 
a multi-line
string.""" # <-- Triple quotes allow for multi-line strings
print(word)
print(sentence)
print(multiline) 

hello
Python is fun!
This is 
a multi-line
string.


### **JavaScript**

In [ ]:
// HTML structure for the game
const gameHTML = `
<div id="game-container" style="display: none;">
    <h2>Match It Game</h2>
    <p>Match the pairs to win!</p>
    <button id="complete-game">Complete Game</button>
</div>
<div id="lesson-content">
    <h2>Lesson Content</h2>
    <p>This content will be visible after you complete the game.</p>
</div>
`;

document.body.innerHTML = gameHTML;

<IPython.core.display.Javascript object>

## Common String Operations
---

## Challenge 2 - Apple Smashing
Click the Apple for a quiz!

<div id="gameArea2" 
     style="position:relative;width:330px;height:230px;border:2px solid #8b5a2b;
            overflow:hidden;background:#fff3e0;display:flex;justify-content:center;
            align-items:center;">

  <div id="cookie2" style="font-size:150px;cursor:pointer;">
    🍎
  </div>

</div>

<p id="question2" style="font-size:18px;font-weight:bold;margin-top:10px;"></p>

<script>
// SUPER EASY QUESTIONS
const questions2 = [
  "PYTHON: Which makes the word uppercase: name.upper() or name.lower()",
  "PYTHON: Which gives the first character: name[0] or name[-1]",
  "JS: Which gives the length: food.length or food.toUpperCase()",
  "JS: Which gives the last character: food[food.length - 1] or food[0]"
];

document.getElementById("cookie2").onclick = () => {
  const cookie = document.getElementById("cookie2");

  // TRUE VIBRATION + 1.5× POP
  cookie.style.transition = "transform 0.15s ease";
  cookie.style.transform = "scale(1.5) translateX(-6px)";
  setTimeout(() => {
    cookie.style.transform = "scale(1.5) translateX(6px)";
  }, 75);
  setTimeout(() => {
    cookie.style.transform = "scale(1)";
  }, 150);

  // Pick a random question
  const q = questions2[Math.floor(Math.random() * questions2.length)];
  document.getElementById("question2").textContent = "Challenge: " + q;
};
</script>

### **Python**

In [9]:
name = "Aneesh"

print(name.upper())     # "ANEESH"
print(name.lower())     # "aneesh"
print(len(name))        # 6
print(name[0])          # "A" (first character)
print(name[-1])         # "h" (last character)
print(name[1:3])        # "ne" (slice)

ANEESH
aneesh
6
A
h
ne


### **JavaScript**

In [10]:
%%javascript

let food = "Pizza"

console.log(food.toUpperCase()); // prints pizza in all upper case
console.log(food.toLowerCase()); // prints pizza in all lower case
console.log(food.length) // prints the number of characters in the word pizza
console.log(food[0]); // prints the first character in pizza
console.log(food[food.length - 1]); // prints the last character in pizza
console.log(food.substring(1,4)); // substring slices the word to get a chunk, the index/numbers are assigned to letters
// pizza = 01234

<IPython.core.display.Javascript object>

## Concatenation (Combining Strings)
---
## Challenge 3 - Orange Popping
Click the Orange for a quiz!

<div id="gameArea3" 
     style="position:relative;width:330px;height:230px;border:2px solid #8b5a2b;
            overflow:hidden;background:#fff3e0;display:flex;justify-content:center;
            align-items:center;">

  <div id="cookieWrapper3" style="cursor:pointer;display:flex;justify-content:center;align-items:center;">
      <div id="cookie3" style="font-size:150px;line-height:1;">
        🍊
      </div>
  </div>

</div>

<p id="question3" style="font-size:18px;font-weight:bold;margin-top:10px;"></p>

<script>
// REAL, EASY CONCATENATION QUESTIONS
const concatQuestions = [
  "PYTHON: Which adds a space: first + \" \" + second or first + second",
  "PYTHON: Which uses an f-string: f\"{first}, {second}!\" or first + second",
  "JS: Which combines strings: first.concat(second) or first - second",
  "JS: Which adds punctuation: first.concat(\", \", second, \"!\") or first.concat(second)"
];

document.getElementById("cookieWrapper3").onclick = () => {
  const cookie = document.getElementById("cookie3");

  // ⭐ 1.5× POP + SHAKE EFFECT
  cookie.style.transition = "transform 0.15s ease";
  cookie.style.transform = "scale(1.5) translateX(-6px)";
  setTimeout(() => {
    cookie.style.transform = "scale(1.5) translateX(6px)";
  }, 75);
  setTimeout(() => {
    cookie.style.transform = "scale(1)";
  }, 150);

  // ⭐ Pick a random question
  const q = concatQuestions[Math.floor(Math.random() * concatQuestions.length)];
  document.getElementById("question3").textContent = "Challenge: " + q;
};
</script>

### **Python**

In [12]:
first = "Hello"
second = "World"

print(first + " " + second)        # "Hello World"
print(first + ", " + second + "!") # "Hello, World!"
print("{} {}".format(first, second))       # "Hello World" (str.format)
print(f"{first}, {second}!")       # "Hello, World!" (f-string) 

Hello World
Hello, World!
Hello World
Hello, World!


### **JavaScript**

In [13]:
%%javascript

let first = "Goodbye";
let second = "Universe";

// Using concat()
console.log(first.concat(" ", second));        // uses concat to combine strings
console.log(first.concat(", ", second, "!"));  // also uses concat but adds a ! at the end
console.log(second.concat(first)); // can use concat to combine strings any way


<IPython.core.display.Javascript object>

## Final Challenge - Fruit Frenzy
Click whatever for a quiz!

<div id="gameArea" 
     style="width:700px;height:450px;border:3px solid #8b5a2b;
            background:#fff3e0;position:relative;margin:auto;overflow:hidden;">

  <div class="fruit" data-fruit="cherry" style="font-size:100px;position:absolute;">🍒</div>
  <div class="fruit" data-fruit="cookie" style="font-size:100px;position:absolute;">🍪</div>
  <div class="fruit" data-fruit="apple"  style="font-size:100px;position:absolute;">🍎</div>

</div>

<p id="finalMessage" style="font-size:20px;font-weight:bold;text-align:center;margin-top:10px;"></p>

<script>
// ⭐ SUPER-EASY, FUN FINAL QUESTIONS
const fruitQuestions = {
  cherry: [
    { q: "Which one is a string: \"hello\" or 5", a: "\"hello\"" },
    { q: "Which one is a string: \"pizza\" or pizza", a: "\"pizza\"" },
    { q: "Which one is a string: \"123\" or 123", a: "\"123\"" }
  ],
  cookie: [
    { q: "Which makes text UPPERCASE: word.upper() or word.lower()", a: "word.upper()" },
    { q: "Which gives the FIRST letter: word[0] or word[-1]", a: "word[0]" },
    { q: "Which counts letters: len(word) or word.upper()", a: "len(word)" }
  ],
  apple: [
    { q: "Which adds a space: first + \" \" + second or first + second", a: "first + \" \" + second" },
    { q: "Which uses an f-string: f\"{first} {second}\" or first + second", a: "f\"{first} {second}\"" },
    { q: "Which joins strings: first.concat(second) or first - second", a: "first.concat(second)" }
  ]
};

// click counters
const clickCounts = { cherry:0, cookie:0, apple:0 };

// bouncing setup
const fruits = document.querySelectorAll(".fruit");
const box = document.getElementById("gameArea");
const FRUIT_SIZE = 100;

fruits.forEach(fruit => {
  // full-box random start
  fruit.x = Math.random() * (box.clientWidth - FRUIT_SIZE);
  fruit.y = Math.random() * (box.clientHeight - FRUIT_SIZE);

  // gentle but free movement
  fruit.vx = (Math.random() * 2 + 1) * (Math.random() < 0.5 ? -1 : 1);
  fruit.vy = (Math.random() * 2 + 1) * (Math.random() < 0.5 ? -1 : 1);

  fruit.style.left = fruit.x + "px";
  fruit.style.top = fruit.y + "px";
});

// animation loop
function animate() {
  fruits.forEach(fruit => {
    fruit.x += fruit.vx;
    fruit.y += fruit.vy;

    // bounce on full box edges
    if (fruit.x <= 0 || fruit.x >= box.clientWidth - FRUIT_SIZE) fruit.vx *= -1;
    if (fruit.y <= 0 || fruit.y >= box.clientHeight - FRUIT_SIZE) fruit.vy *= -1;

    fruit.style.left = fruit.x + "px";
    fruit.style.top = fruit.y + "px";
  });

  requestAnimationFrame(animate);
}
animate();

// click behavior
fruits.forEach(fruit => {
  fruit.onclick = () => {
    const name = fruit.dataset.fruit;

    // 1.5× POP + SHAKE
    fruit.style.transition = "transform 0.15s ease";
    fruit.style.transform = "scale(1.5) translateX(-6px)";
    setTimeout(() => fruit.style.transform = "scale(1.5) translateX(6px)", 75);
    setTimeout(() => fruit.style.transform = "scale(1)", 150);

    clickCounts[name]++;

    if (clickCounts[name] % 5 === 0) {
      askQuestion(name);
    }
  };
});

function askQuestion(fruitName) {
  const qSet = fruitQuestions[fruitName];
  const item = qSet[Math.floor(Math.random() * qSet.length)];

  while (true) {
    const answer = prompt(
      "Final Challenge:\n\n" + 
      item.q + 
      "\n\nType your answer, or click Cancel to skip."
    );

    // ⭐ Cancel now WORKS as skip
    if (answer === null) {
      document.getElementById("finalMessage").textContent = "Skipped!";
      break;
    }

    if (answer.trim() === item.a) {
      document.getElementById("finalMessage").textContent = "Correct!";
      break;
    } else {
      document.getElementById("finalMessage").textContent = "Try again!";
    }
  }
}
</script>

### Practice Questions!

1. What will the command **concat** do in JavaScript?
2. Is the **substring** command used in?
3. How do you do **multiline** strings in JavaScript?

---
### MCQ
Go as fast as you can to try to win!

[Click here for qualifying](https://whitelunarium.github.io/Aneesh_2026/mcq_strings)

[Click here for final](https://whitelunarium.github.io/Aneesh_2026/mcq_strings2)